# Lektion 11 - Model Context Protocol (MCP)

Das **Model Context Protocol (MCP)** ist ein offener Standard, der Agenten ermöglicht, Werkzeuge, Ressourcen und Datenquellen zur Laufzeit dynamisch zu entdecken und zu nutzen. Anstatt Werkzeuge in einen Agenten fest zu kodieren, erlaubt MCP Agenten, sich mit externen Servern zu verbinden, die Funktionen bedarfsgerecht bereitstellen.

In dieser Lektion lernst du:
- Was MCP ist und warum es für Agentensysteme wichtig ist
- Wie die Client-Server-Architektur von MCP funktioniert
- Wie man Agenten baut, die die Tool-Erkennung im MCP-Stil verwenden


## Einrichtung

**Voraussetzungen:**
- Azure AI Foundry-Projekt mit einem bereitgestellten Modell
- Führen Sie `az login` zur Authentifizierung aus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Was ist das Model Context Protocol (MCP)?

MCP definiert eine standardisierte Methode, mit der KI-Agenten externe Werkzeuge und Datenquellen entdecken und mit ihnen interagieren können:

- **MCP Server**: Stellt Werkzeuge, Ressourcen und Prompts über ein standardisiertes Protokoll bereit
- **MCP Client**: Die Agenten-Laufzeitumgebung, die sich mit Servern verbindet und verfügbare Fähigkeiten entdeckt
- **Dynamic Discovery**: Agenten benötigen keine fest verdrahteten Werkzeuge — sie entdecken zur Laufzeit, was verfügbar ist

Dies ist besonders nützlich beim Aufbau erweiterbarer Agentensysteme, in denen neue Fähigkeiten hinzugefügt werden können, ohne den Agentencode zu verändern.


## Wie MCP funktioniert

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Der Agent (MCP client) verbindet sich mit einem MCP server
2. Der Server antwortet mit einer Liste verfügbarer Tools und deren Schemata
3. Der Agent kann dann während seines Denkprozesses jedes entdeckte Tool aufrufen
4. Ergebnisse fließen über dasselbe Protokoll zurück


## Simulation der MCP-Tool-Erkennung

Da ein echter MCP-Server einen laufenden Serverprozess benötigt, demonstrieren wir das Muster mithilfe von `@tool`-Funktionen, die nachahmen, was ein an MCP angeschlossener Unterkunftsdienst bereitstellen würde.

In der Produktionsumgebung würden diese Tools dynamisch von einem MCP-Server entdeckt werden, anstatt lokal definiert zu sein.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Einen Agenten mit MCP-ähnlichen Werkzeugen erstellen


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP in der Produktion

In einer Produktionsumgebung ermöglicht MCP leistungsstarke Muster:

- **Dynamische Tool-Erkennung**: Agenten verbinden sich mit MCP-Servern und entdecken Tools zur Laufzeit
- **Entkoppelte Architektur**: Tool-Anbieter können ihre Tools unabhängig vom Agenten aktualisieren
- **Organisationenübergreifende Freigabe**: Teams können Fähigkeiten über MCP-Server bereitstellen, die von jedem Agenten genutzt werden können
- **Unterstützung durch das Microsoft Agent Framework**: MAF verfügt über integrierte MCP-Client-Unterstützung über die `mcp`-Integration

Um einen echten MCP-Server mit MAF zu verwenden, verbinden Sie sich über `hosted_mcp_tool()` oder die MCP-Client-Integration.

**Mehr erfahren:**
- [MCP-Spezifikation](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP-Unterstützung](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Zusammenfassung

In dieser Lektion haben Sie gelernt:
- **MCP** ist ein offener Standard zur dynamischen Entdeckung von Tools zwischen Agenten und Tool-Anbietern
- Die **Client-Server-Architektur** ermöglicht es Agenten, Fähigkeiten zur Laufzeit zu entdecken
- MCP ermöglicht **erweiterbare, entkoppelte Agentensysteme**, bei denen Tools ohne Codeänderungen hinzugefügt werden können
- Microsoft Agent Framework bietet **integrierte MCP-Unterstützung** für den Produktivbetrieb


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Haftungsausschluss:
Dieses Dokument wurde mithilfe des KI-Übersetzungsdienstes Co-op Translator (https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatische Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache ist als maßgebliche Quelle zu betrachten. Bei wichtigen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die sich aus der Verwendung dieser Übersetzung ergeben.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
